In [ ]:
import os
import random
import shutil

# Definir los directorios de origen y destino
src_folder = "spectrums"
dst_folder = "spectrums training 50k"

# Crear la carpeta de destino si no existe
if not os.path.exists(dst_folder):
    os.makedirs(dst_folder)

# Listar todos los archivos .fits en el directorio de origen
fits_files = [f for f in os.listdir(src_folder) if f.lower().endswith(".fits")]

# Seleccionar 100000 archivos aleatorios
selected_files = random.sample(fits_files, 100000)

# Copiar cada archivo seleccionado a la carpeta de destino
for file in selected_files:
    src_path = os.path.join(src_folder, file)
    dst_path = os.path.join(dst_folder, file)
    shutil.copy(src_path, dst_path)
    print(f"Copiado: {file}")

print("Copia completada.")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

# Lista para almacenar los valores de redshift
redshift_values = []

# Recorrer todos los archivos .fits en la carpeta
for filename in os.listdir(dst_folder):
    if filename.lower().endswith('.fits'):
        file_path = os.path.join(dst_folder, filename)
        try:
            with fits.open(file_path) as hdul:
                # Extraer redshift (se asume que está en la extensión 2, campo "Z")
                redshift = hdul[2].data["Z"][0]
                redshift_values.append(redshift)
                # print(f"Procesado {filename}: redshift = {redshift}")
        except Exception as e:
            print(f"Error procesando {filename}: {e}")

# Convertir la lista a un arreglo de numpy
redshift_array = np.array(redshift_values)

# Definir los bins con intervalos de 0.1
bins = np.arange(0, redshift_array.max() + 0.1, 0.1)

plt.figure(figsize=(18,16))
plt.hist(redshift_array, bins=bins, color='skyblue', edgecolor='black')
plt.xlabel("Redshift")
plt.ylabel("Número de archivos")
plt.title("Histograma de redshifts (bin width = 0.1)")
plt.xlim(0, 7.1)
plt.show()

In [ ]:
import os
import time
from astropy.io import fits

def try_remove(file_path, attempts=3, delay=5):
    for attempt in range(attempts):
        try:
            os.remove(file_path)
            return True
        except OSError as e:
            # Verificar si el error es WinError 32 (archivo en uso)
            if hasattr(e, 'winerror') and e.winerror == 32:
                print(f"Archivo {file_path} en uso, reintentando en {delay} segundos... (intento {attempt+1}/{attempts})")
                time.sleep(delay)
            else:
                print(f"Error removiendo {file_path}: {e}")
                break
    return False

# Definir el directorio que contiene los archivos FITS
src_folder = "spectrums training 100k"

# Recorrer todos los archivos FITS en el directorio
for file in os.listdir(src_folder):
    if file.lower().endswith(".fits"):
        file_path = os.path.join(src_folder, file)
        try:
            with fits.open(file_path) as hdul:
                # Se asume que el redshift se encuentra en la extensión 2, campo "Z"
                redshift = hdul[2].data["Z"][0]
            # Verificar si el redshift está en el rango 0.1 <= z < 1
            if 3.0 <= redshift < 3.1:
                if try_remove(file_path):
                    print(f"Eliminado: {file}")
                else:
                    print(f"No se pudo eliminar: {file}")
        except Exception as e:
            print(f"Error procesando {file}: {e}")

print("Proceso de eliminación completado.")